In [0]:
spark.sql("USE CATALOG konami_catalog")
spark.sql("USE SCHEMA retail_schema")

DataFrame[]

Load Bronze Tables

In [0]:
customers_df = spark.table("konami_catalog.retail_schema.bronze_customers")
orders_df = spark.table("konami_catalog.retail_schema.bronze_orders")
products_df = spark.table("konami_catalog.retail_schema.bronze_products")
order_items_df = spark.table("konami_catalog.retail_schema.bronze_order_items")
regional_df = spark.table("konami_catalog.retail_schema.bronze_regional_market_targets")

Data Cleaning - Silver Tables

1. Customer Silver Table

In [0]:
from pyspark.sql.functions import *

# Standardize text fields
customers_clean = customers_df \
    .withColumn("gender",
        when(col("gender").isin("M", "Male", "male", "MALE"), "Male")
        .when(col("gender").isin("F", "Female", "female"), "Female")
        .otherwise(None)
    ) \
    .withColumn("country", initcap(col("country"))) \
    .withColumn("customer_segment", initcap(col("customer_segment"))) \
    .withColumn("customer_name", initcap(col("customer_name")))

# Fix date formats
customers_clean = customers_clean \
    .withColumn("signup_date",
        coalesce(
            try_to_date(col("signup_date"), "yyyy-MM-dd"),
            try_to_date(col("signup_date"), "yyyy/MM/dd")
        )
    )

# Remove invalid records
customers_clean = customers_clean.filter(
    (col("age").isNotNull()) & 
    (col("age") > 0) & 
    (col("gender").isNotNull()) &
    (col("customer_segment").isNotNull()) &
    (col("signup_date").isNotNull())
)

# Save Silver table
customers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_customers")

display(customers_clean)

customer_id,customer_name,age,gender,country,region_id,signup_date,customer_segment
1,Arafat Mahmood,26,Male,Nigeria,1,2026-01-10,Premium
3,Fabian Akaeze,35,Male,Singapore,6,2026-01-20,Premium
5,Samuel Mensah,42,Male,Ghana,1,2026-02-10,Premium
6,Priya Sharma,31,Female,India,5,2026-01-25,Standard
8,Karishma Shaik,23,Female,South Africa,4,2026-03-05,Standard
9,John Smith,38,Male,Kenya,2,2026-02-01,Premium
11,Grace Wanjiku,27,Female,Kenya,2,2026-02-20,Standard
12,Daniel Kim,33,Male,Singapore,6,2026-02-18,Premium
13,Orukome Egbuwoku,24,Female,Nigeria,1,2026-03-02,Standard
14,Michael Lee,41,Male,Malaysia,7,2026-01-30,Premium


In [0]:
customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_customers")


2. Product Silver Table

In [0]:
from pyspark.sql.functions import *

# Standardize text columns
products_clean = products_df \
    .withColumn("category", initcap(col("category"))) \
    .withColumn("supplier_country", initcap(col("supplier_country"))) \
    .withColumn("product_name", initcap(col("product_name")))

# Cast incorrect data types
products_clean = products_clean \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("stock_quantity", col("stock_quantity").cast("integer"))

# Remove invalid records
products_clean = products_clean.filter(
    (col("product_id").isNotNull()) &
    (col("product_name").isNotNull()) &
    (col("category").isNotNull()) &
    (col("price").isNotNull()) &
    (col("stock_quantity").isNotNull()) &
    (col("price") > 0) &
    (col("stock_quantity") >= 0) &
    (col("supplier_country").isNotNull())
)

# Save Silver Products Table
products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_products")


display(products_clean)

product_id,product_name,category,price,stock_quantity,supplier_country
101,Gaming Mouse,Electronics,29.99,100,China
102,Wireless Headset,Electronics,49.99,50,China
104,Laptop Stand,Accessories,25.5,75,India
105,Mechanical Keyboard,Electronics,79.99,40,China
106,Usb-c Cable,Accessories,9.99,200,Vietnam
107,External Hard Drive,Electronics,120.0,30,Thailand
108,Monitor 24 Inch,Electronics,199.99,20,China
109,Office Chair,Furniture,150.75,15,Malaysia
111,Tablet,Electronics,250.0,25,China
114,Webcam,Electronics,70.0,35,Vietnam


In [0]:
products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_products")

3. Order Silver Table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Standardize text columns

orders_clean = orders_df \
    .withColumn("shipping_country", initcap(col("shipping_country"))) \
    .withColumn("order_status",
        when(lower(col("order_status")) == "delivered", "Delivered")
        .when(lower(col("order_status")) == "pending", "Pending")
        .when(lower(col("order_status")) == "cancelled", "Cancelled")
        .otherwise(None)
    ) \
    .withColumn("payment_method",
        initcap(col("payment_method"))
    )

# Convert incorrect data types
orders_clean = orders_clean \
    .withColumn(
        "shipping_days",
        expr("try_cast(shipping_days as int)")
    )

# Fix date formats
orders_clean = orders_clean \
    .withColumn(
        "order_date",
        coalesce(
            try_to_date(col("order_date"), "yyyy-MM-dd"),
            try_to_date(col("order_date"), "yyyy/MM/dd")
        )
    )

# Remove invalid records
orders_clean = orders_clean.filter(
    (col("order_id").isNotNull()) &
    (col("customer_id").isNotNull()) &
    (col("order_date").isNotNull()) &
    (col("shipping_country").isNotNull()) &
    (col("payment_method").isNotNull()) &
    (col("shipping_days").isNotNull()) &
    (col("shipping_days") > 0)
)

# 5. Validate customer relationship
customers_df = spark.table(
    "konami_catalog.retail_schema.silver_customers"
)


orders_clean = orders_clean.join(
    customers_df.select("customer_id"),
    on="customer_id",
    how="inner"
)

# 6. Save Silver Orders Table
orders_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_orders")


display(orders_clean)

customer_id,order_id,order_date,shipping_country,payment_method,order_status,shipping_days
1,1001,2026-02-01,Nigeria,Card,Delivered,5
3,1003,2026-02-10,Singapore,Paypal,Pending,3
5,1005,2026-02-18,Ghana,Card,Delivered,6
6,1006,2026-02-20,India,Upi,Delivered,4
8,1008,2026-02-25,South Africa,Card,Pending,5
9,1009,2026-02-28,Kenya,Card,Delivered,6
12,1012,2026-03-03,Singapore,Paypal,Pending,4
13,1013,2026-03-04,Nigeria,Card,Delivered,7
14,1014,2026-03-05,Malaysia,Card,Cancelled,2
18,1018,2026-03-09,Kenya,Card,Delivered,4


In [0]:
orders_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_orders")

4. Order Items Silver Table

In [0]:
from pyspark.sql.functions import *

# Convert incorrect data types
order_items_clean = order_items_df \
    .withColumn(
        "quantity",
        expr("try_cast(quantity as int)")
    ) \
    .withColumn(
        "unit_price",
        expr("try_cast(unit_price as double)")
    ) \
    .withColumn(
        "discount",
        expr("try_cast(discount as double)")
    )

# Remove invalid records
order_items_clean = order_items_clean.filter(
    (col("order_id").isNotNull()) &
    (col("product_id").isNotNull()) &
    (col("quantity").isNotNull()) &
    (col("unit_price").isNotNull()) &
    (col("quantity") > 0) &
    (col("unit_price") > 0)
)

# Handle missing discounts
order_items_clean = order_items_clean.fillna({
    "discount": 0.0
})

# Validate foreign keys
orders_df = spark.table(
    "konami_catalog.retail_schema.silver_orders"
)

products_df = spark.table(
    "konami_catalog.retail_schema.silver_products"
)

# Keep only valid order IDs
order_items_clean = order_items_clean.join(
    orders_df.select("order_id"),
    on="order_id",
    how="inner"
)

# Keep only valid product IDs
order_items_clean = order_items_clean.join(
    products_df.select("product_id"),
    on="product_id",
    how="inner"
)

# Save Silver Order Items Table
order_items_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_order_items")


display(order_items_clean)

product_id,order_id,quantity,unit_price,discount
101,1001,2,29.99,0.0
105,1001,1,79.99,5.0
108,1003,1,199.99,0.0
111,1005,1,250.0,0.0
120,1008,3,12.5,0.0
121,1009,1,500.0,0.0
101,1014,2,29.99,0.0
105,1018,1,79.99,0.0
106,1019,4,9.99,0.0
108,1021,2,199.99,15.0


In [0]:
order_items_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_order_items")

5. Regional Market Targets Silver table

In [0]:
from pyspark.sql.functions import *

# Standardize text columns
regional_clean = regional_df \
    .withColumn("region_name", initcap(col("region_name"))) \
    .withColumn("country_group", initcap(col("country_group")))

# Convert numeric data types
regional_clean = regional_clean \
    .withColumn(
        "target_customers",
        expr("try_cast(target_customers as int)")
    ) \
    .withColumn(
        "target_revenue",
        expr("try_cast(target_revenue as double)")
    )

# Remove incomplete records
regional_clean = regional_clean.filter(
    (col("region_id").isNotNull()) &
    (col("region_name").isNotNull()) &
    (col("country_group").isNotNull()) &
    (col("target_customers").isNotNull()) &
    (col("target_revenue").isNotNull())
)

# 4. Save Silver Regional Table
regional_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "konami_catalog.retail_schema.silver_regional_market_targets"
    )


display(regional_clean)

region_id,region_name,country_group,target_customers,target_revenue
1,West Africa,Africa,50000,500000.0
2,East Africa,Africa,40000,400000.0
4,Southern Africa,Africa,35000,350000.0
5,India,South Asia,90000,900000.0
6,Singapore,Southeast Asia,60000,700000.0


In [0]:
regional_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("konami_catalog.retail_schema.silver_regional_market_targets")

#### Gold Table 1 - Regional Market Performance

This table combines customer, order, order item, and regional target data to evaluate Konami's market penetration across different regions.

The analysis compares actual revenue generated in each region against predefined expansion targets.

In [0]:
from pyspark.sql.functions import *


# Load Silver tables

customers_df = spark.table(
    "konami_catalog.retail_schema.silver_customers"
)

orders_df = spark.table(
    "konami_catalog.retail_schema.silver_orders"
)

order_items_df = spark.table(
    "konami_catalog.retail_schema.silver_order_items"
)

regional_df = spark.table(
    "konami_catalog.retail_schema.silver_regional_market_targets"
)


# Calculate revenue per order item

sales_df = order_items_df.withColumn(
    "revenue",
    (col("quantity") * col("unit_price")) - col("discount")
)


# Join tables

regional_performance = (
    sales_df
    .join(
        orders_df,
        "order_id"
    )
    .join(
        customers_df,
        "customer_id"
    )
    .join(
        regional_df,
        "region_id"
    )
)


# Aggregate by region

gold_regional_market = (
    regional_performance
    .groupBy(
        "region_id",
        "region_name",
        "country_group",
        "target_revenue"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        round(sum("revenue"),2).alias("actual_revenue")
    )
    .withColumn(
        "revenue_target_percentage",
        round(
            (col("actual_revenue") / col("target_revenue")) * 100,
            2
        )
    )
)


display(gold_regional_market)

region_id,region_name,country_group,target_revenue,total_orders,unique_customers,actual_revenue,revenue_target_percentage
1,West Africa,Africa,500000.0,4,3,949.9,0.19
2,East Africa,Africa,400000.0,2,2,579.99,0.14
6,Singapore,Southeast Asia,700000.0,2,2,277.45,0.04
4,Southern Africa,Africa,350000.0,1,1,37.5,0.01


In [0]:
gold_regional_market.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "konami_catalog.retail_schema.gold_regional_market_performance"
    )

#### Gold Table 2 - Product Category Performance

This table analyzes product sales performance by combining product information with transaction details.

It identifies categories generating the highest revenue and sales volume.

In [0]:
products_df = spark.table(
    "konami_catalog.retail_schema.silver_products"
)


order_items_df = spark.table(
    "konami_catalog.retail_schema.silver_order_items"
)


product_sales = (
    order_items_df
    .withColumn(
        "revenue",
        (col("quantity") * col("unit_price")) - col("discount")
    )
    .join(
        products_df,
        "product_id"
    )
)


gold_product = (
    product_sales
    .groupBy(
        "category"
    )
    .agg(
        countDistinct("product_id").alias("total_products"),
        sum("quantity").alias("units_sold"),
        round(sum("revenue"),2).alias("total_revenue"),
        round(avg("unit_price"),2).alias("average_price")
    )
)


display(gold_product)

category,total_products,units_sold,total_revenue,average_price
Electronics,6,13,1744.91,159.99
Accessories,3,15,159.91,11.0


In [0]:
gold_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "konami_catalog.retail_schema.gold_product_performance"
    )

#### Gold Table 3 - Shipping Analysis

This table evaluates shipping performance by country.

Metrics include average delivery time, order volume, and delivery success rate.

In [0]:
orders_df = spark.table(
    "konami_catalog.retail_schema.silver_orders"
)


gold_shipping = (
    orders_df
    .groupBy(
        "shipping_country"
    )
    .agg(
        count("order_id").alias("total_orders"),
        round(avg("shipping_days"),2).alias("average_shipping_days"),
        
        sum(
            when(
                col("order_status")=="Delivered",
                1
            ).otherwise(0)
        ).alias("delivered_orders"),
        
        sum(
            when(
                col("order_status")=="Cancelled",
                1
            ).otherwise(0)
        ).alias("cancelled_orders")
    )
    .withColumn(
        "delivery_success_rate",
        round(
            (col("delivered_orders") / col("total_orders")) * 100,
            2
        )
    )
)


display(gold_shipping)

shipping_country,total_orders,average_shipping_days,delivered_orders,cancelled_orders,delivery_success_rate
Nigeria,3,5.67,2,0,66.67
Singapore,3,4.33,1,0,33.33
Ghana,2,6.0,2,0,100.0
India,1,4.0,1,0,100.0
South Africa,1,5.0,0,0,0.0
Kenya,2,5.0,2,0,100.0
Malaysia,1,2.0,0,1,0.0


In [0]:
gold_shipping.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "konami_catalog.retail_schema.gold_shipping_analysis"
    )